# Drive and utilities


In [1]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

!pip install numpy pandas scipy scikit-learn sklearn tqdm cudf cupy

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy import stats

## Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Using device: cpu


# Functions

In [53]:
import pandas as pd

def trim_records(citations, patent_ids):
  """
  Input:
    - citations Pandas DataFrame with correct column names
    - list of patent IDs from the matched sample (treated and control)
  Output:
    -trimmed citations Pandas DataFrame
  """
  citations['patent_id'] = citations['patent_id'].astype(str)
  citations = citations[citations['patent_id'].isin(patent_ids)]
  print("Trimmed citations shape:", citations.shape)

  return citations


def precompute_quarterly_unique_records(citations):
  """
  Input:
    - Trimmed citations Pandas DataFrame
  Output:
    - dictionary mapping patent_id -> {quarter: count}
  """

  citations[['cpc_group_agg', 'cpc_group_spec']] = citations['cpc_group'].str.split('/', expand = True)
  citations['cpc_group_agg'].value_counts()

  # Manually compute a quarter string: "YYYYQX"
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
  citations['year'] = citations['citation_date'].dt.year
  citations['month'] = citations['citation_date'].dt.month

  ## Compute quarter as integer: ((month-1)//3 + 1)
  citations['qtr'] = ((citations['month'] - 1) // 3 + 1).astype(str)
  citations['citation_quarter'] = citations['year'].astype(str) + "Q" + citations['qtr']

  # Group by patent_id and citation_quarter to count unique CPCs and assignees per quarter.
  unique_cpc = citations.groupby(['patent_id', 'citation_quarter'])['cpc_group_agg'].nunique()
  unique_assignee = citations.groupby(['patent_id', 'citation_quarter'])['disambig_assignee_organization'].nunique()

  unique_cpc_pd = unique_cpc.unstack(fill_value=0)
  unique_assignee_pd = unique_assignee.unstack(fill_value=0)

  # Build a dictionary mapping patent_id -> {quarter: count}
  unique_cpcs_dict = {}
  for pid, row in unique_cpc_pd.iterrows():
      unique_cpcs_dict[pid] = row.to_dict()

  unique_assignees_dict = {}
  for pid, row in unique_assignee_pd.iterrows():
      unique_assignees_dict[pid] = row.to_dict()

  return unique_cpcs_dict, unique_assignees_dict

def get_unique_ids(df):

  df['treated_id'] = df['treated_id'].astype(str)
  df['control_id'] = df['control_id'].astype(str)

  print(f"Sample size : {len(df)}.")

  ## Get IDs in the samples
  patent_ids = []


  patent_ids.extend(df['treated_id'].unique())
  patent_ids.extend(df['control_id'].unique())

  patent_ids = np.unique(patent_ids)
  return patent_ids






# Load data

In [20]:
#@title Citations - filing

## Load citations as Pandas DataFrame from parquet & rename.
citations = pd.read_pickle("/content/drive/MyDrive/PhD Data/08 Citations/03 Patent citations - raw, filing.pickle")
citations.rename(columns={'patent_id': 'citedby_patent_id', 'citation_patent_id':'patent_id', 'filing_date':'citation_date'}, inplace = True)
citations['patent_id'] = citations['patent_id'].astype(str)
citations['citedby_patent_id'] = citations['citedby_patent_id'].astype(str)

citations.set_index('citedby_patent_id', inplace = True)

In [21]:
#@title Patents & CPC Current & Assignees (TSV files)

import pandas as pd

# Patents
patents = pd.read_csv("/content/drive/MyDrive/PhD Data/00 Raw data from PV/g_patent.tsv/g_patent.tsv", sep = '\t')
patents['patent_id'] = patents['patent_id'].astype(str)

# CPC Current
cpc_current = pd.read_csv("/content/drive/MyDrive/PhD Data/00 Raw data from PV/g_cpc_current.tsv/g_cpc_current.tsv", sep = '\t')
cpc_current['patent_id'] = cpc_current['patent_id'].astype(str)
cpc_current = cpc_current[cpc_current['cpc_sequence'] == 0]

# Assignee Disambiguated
assignee_disambiguated = pd.read_csv("/content/drive/MyDrive/PhD Data/00 Raw data from PV/g_assignee_disambiguated.tsv", sep = '\t')
assignee_disambiguated['patent_id'] = assignee_disambiguated['patent_id'].astype(str)
assignee_disambiguated = assignee_disambiguated[assignee_disambiguated['assignee_sequence'] == 0]
assignee_disambiguated = assignee_disambiguated[~assignee_disambiguated['disambig_assignee_organization'].isna()]

cols = ['patent_id', 'disambig_assignee_organization']
assignee_disambiguated = assignee_disambiguated[cols]

# Combine
patents.set_index('patent_id', inplace = True)
cpc_current.set_index('patent_id', inplace = True)
assignee_disambiguated.set_index('patent_id', inplace = True)

patents = patents.join(cpc_current, how = 'inner')
patents = patents.join(assignee_disambiguated, how = 'inner')

# Filter
cols = ['cpc_group', 'cpc_subclass', 'disambig_assignee_organization']
patents = patents[cols]

# Combine
citations = citations.join(patents, how = 'inner')

<ipython-input-21-3aedd55c3ded>:6: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  patents = pd.read_csv("/content/drive/MyDrive/PhD Data/00 Raw data from PV/g_patent.tsv/g_patent.tsv", sep = '\t')


# Combine with unique records


In [54]:
#@title Top Tech - M&A

# Sample
sample = pd.read_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, MA, filing date.csv")

# Convert the acq_date column to datetime, then to a quarterly period
sample['acq_quarter'] = pd.to_datetime(sample['acq_date']).dt.to_period('Q')

# Add the quarter shifter to compute the actual quarter
sample['actual_quarter'] = (sample['acq_quarter'] + sample['quarter']).astype(str)

# Get unique IDs
patent_ids = get_unique_ids(sample)

# Trim citations
records = trim_records(citations, patent_ids)

# Get Unique Dicts
unique_cpcs_dict, unique_assignees_dict = precompute_quarterly_unique_records(records)

# initialize lists to hold values
treated_cpc_vec = []
treated_assignee_vec = []
control_cpc_vec = []
control_assignee_vec = []

for idx, row in tqdm(sample.iterrows(), total = len(sample), desc = "Looping over rows"):
  # Actual quarter
  actual_quarter = row['actual_quarter']
  q_period = pd.Period(actual_quarter, freq='Q')
  last_quarter = pd.Period("2024Q4", freq = 'Q')

  # Treated patent
  treated_id = str(row['treated_id'])

  ## CPCs
  if treated_id in unique_cpcs_dict and actual_quarter in unique_cpcs_dict[treated_id]:
    treated_cpc_vec.append(unique_cpcs_dict[treated_id][row['actual_quarter']])
  else:
    treated_cpc_vec.append(np.nan if q_period > last_quarter else 0)

  ## Assignee
  if treated_id in unique_assignees_dict and actual_quarter in unique_assignees_dict[treated_id]:
    treated_assignee_vec.append(unique_assignees_dict[treated_id][row['actual_quarter']])
  else:
    treated_assignee_vec.append(np.nan if q_period > last_quarter else 0)

  # Control patent
  control_id = str(row['control_id'])

  ## CPCs
  if control_id in unique_cpcs_dict and actual_quarter in unique_cpcs_dict[control_id]:
    control_cpc_vec.append(unique_cpcs_dict[control_id][row['actual_quarter']])
  else:
    control_cpc_vec.append(np.nan if q_period > last_quarter else 0)

  ## Assignee
  if control_id in unique_assignees_dict and actual_quarter in unique_assignees_dict[control_id]:
    control_assignee_vec.append(unique_assignees_dict[control_id][row['actual_quarter']])
  else:
    control_assignee_vec.append(np.nan if q_period > last_quarter else 0)



# Check if the length of the list matches the number of rows in the DataFrame
if len(treated_cpc_vec) == len(sample) & len(treated_assignee_vec) == len(sample):
    sample['unique_cpc_treated'] = treated_cpc_vec
    sample['unique_assignee_treated'] = treated_assignee_vec

else:
    raise ValueError("The list length does not match the number of rows in the DataFrame.")

if len(control_cpc_vec) == len(sample) and len(control_assignee_vec) == len(sample):
    sample['unique_cpc_control'] = control_cpc_vec
    sample['unique_assignee_control'] = control_assignee_vec
else:
    raise ValueError("The list length does not match the number of rows in the DataFrame.")

sample.head(20)

## Save
sample.to_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, MA, filing date, enhanced.csv", index = False)



<ipython-input-54-ed11117989ab>:4: DtypeWarning: Columns (26,27,28,29,30,31,32,43,48,51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  sample = pd.read_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, MA, filing date.csv")


Sample size : 537390.
Trimmed citations shape: (456117, 6)


<ipython-input-53-34ef688c143c>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations[['cpc_group_agg', 'cpc_group_spec']] = citations['cpc_group'].str.split('/', expand = True)
<ipython-input-53-34ef688c143c>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations[['cpc_group_agg', 'cpc_group_spec']] = citations['cpc_group'].str.split('/', expand = True)
<ipython-input-53-34ef688c143c>:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[r

In [64]:
#@title Top Tech - Off Deal
from tqdm import tqdm

# Sample
sample = pd.read_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, OD, filing date.csv")

# Convert the acq_date column to datetime, then to a quarterly period
sample['acq_quarter'] = pd.to_datetime(sample['acq_date']).dt.to_period('Q')

# Add the quarter shifter to compute the actual quarter
sample['actual_quarter'] = (sample['acq_quarter'] + sample['quarter']).astype(str)

# Get unique IDs
patent_ids = get_unique_ids(sample)

# Trim citations
records = trim_records(citations, patent_ids)

# Get Unique Dicts
unique_cpcs_dict, unique_assignees_dict = precompute_quarterly_unique_records(records)

# initialize lists to hold values
treated_cpc_vec = []
treated_assignee_vec = []
control_cpc_vec = []
control_assignee_vec = []

for idx, row in tqdm(sample.iterrows(), total = len(sample), desc = "Looping over rows"):
  # Actual quarter
  actual_quarter = row['actual_quarter']
  q_period = pd.Period(actual_quarter, freq='Q')
  last_quarter = pd.Period("2024Q4", freq = 'Q')

  # Treated patent
  treated_id = str(row['treated_id'])

  ## CPCs
  if treated_id in unique_cpcs_dict and actual_quarter in unique_cpcs_dict[treated_id]:
    treated_cpc_vec.append(unique_cpcs_dict[treated_id][row['actual_quarter']])
  else:
    treated_cpc_vec.append(np.nan if q_period > last_quarter else 0)

  ## Assignee
  if treated_id in unique_assignees_dict and actual_quarter in unique_assignees_dict[treated_id]:
    treated_assignee_vec.append(unique_assignees_dict[treated_id][row['actual_quarter']])
  else:
    treated_assignee_vec.append(np.nan if q_period > last_quarter else 0)

  # Control patent
  control_id = str(row['control_id'])

  ## CPCs
  if control_id in unique_cpcs_dict and actual_quarter in unique_cpcs_dict[control_id]:
    control_cpc_vec.append(unique_cpcs_dict[control_id][row['actual_quarter']])
  else:
    control_cpc_vec.append(np.nan if q_period > last_quarter else 0)

  ## Assignee
  if control_id in unique_assignees_dict and actual_quarter in unique_assignees_dict[control_id]:
    control_assignee_vec.append(unique_assignees_dict[control_id][row['actual_quarter']])
  else:
    control_assignee_vec.append(np.nan if q_period > last_quarter else 0)



# Check if the length of the list matches the number of rows in the DataFrame
if len(treated_cpc_vec) == len(sample) & len(treated_assignee_vec) == len(sample):
    sample['unique_cpc_treated'] = treated_cpc_vec
    sample['unique_assignee_treated'] = treated_assignee_vec

else:
    raise ValueError("The list length does not match the number of rows in the DataFrame.")

if len(control_cpc_vec) == len(sample) and len(control_assignee_vec) == len(sample):
    sample['unique_cpc_control'] = control_cpc_vec
    sample['unique_assignee_control'] = control_assignee_vec
else:
    raise ValueError("The list length does not match the number of rows in the DataFrame.")

sample.head(20)

## Save
sample.to_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, OD, filing date, enhanced.csv", index = False)



<ipython-input-64-93948e205cc3>:5: DtypeWarning: Columns (26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  sample = pd.read_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, OD, filing date.csv")


Sample size : 455175.
Trimmed citations shape: (285847, 6)


<ipython-input-53-34ef688c143c>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations[['cpc_group_agg', 'cpc_group_spec']] = citations['cpc_group'].str.split('/', expand = True)
<ipython-input-53-34ef688c143c>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations[['cpc_group_agg', 'cpc_group_spec']] = citations['cpc_group'].str.split('/', expand = True)
<ipython-input-53-34ef688c143c>:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[r

In [67]:
x = sample[sample['treated_id'] == '5943171']

In [69]:
x.head(45)

,treated_id,control_id,match_id,lambda,mahalanobis_distance,cosine_distance,hybrid_distance,quarter,citations_treated,citations_control,...,resold_date,modnote,assignor,_merge,acq_quarter,actual_quarter,unique_cpc_treated,unique_assignee_treated,unique_cpc_control,unique_assignee_control
30450,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,-1,3,14,...,NaN,NaN,IBM,Master only (1),2011Q2,2011Q1,1,2,6,2
30451,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,-2,4,7,...,NaN,NaN,IBM,Master only (1),2011Q2,2010Q4,2,1,4,1
30452,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,-3,1,2,...,NaN,NaN,IBM,Master only (1),2011Q2,2010Q3,1,1,2,2
30453,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,-4,0,6,...,NaN,NaN,IBM,Master only (1),2011Q2,2010Q2,0,0,5,2
30454,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,0,3,0,...,NaN,NaN,IBM,Master only (1),2011Q2,2011Q2,3,1,0,0
30455,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,1,17,5,...,NaN,NaN,IBM,Master only (1),2011Q2,2011Q3,6,4,2,3
30456,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,10,2,0,...,NaN,NaN,IBM,Master only (1),2011Q2,2013Q4,1,1,0,0
30457,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,11,2,3,...,NaN,NaN,IBM,Master only (1),2011Q2,2014Q1,1,2,2,2
30458,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,12,1,1,...,NaN,NaN,IBM,Master only (1),2011Q2,2014Q2,1,1,1,1
30459,5943171,5973760,34,0.00,11.68239,0.114746,0.114746,13,5,2,...,NaN,NaN,IBM,Master only (1),2011Q2,2014Q3,2,2,1,1
